In [70]:
# import packages and data

import pandas as pd
import os

path_data = 'C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/output'
fname = 'survey_effluent.xlsx'
sname = 'Sheet1'

path_out = 'C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/output'

df = pd.read_excel(os.path.join(path_data, fname), sheet_name=sname)

print(df['PREDICTED_WRRF_TT_CODE'].unique())
#[columns for columns in df.columns]

<StringArray>
[   '*AG2', '*B6, *B',  '*AEF2e',    '*AE5',    '*AF2',    '*CE5',    '*B1e',
    '*EF1',    '*A1e',    '*AF1',     '*A1',       nan,   '*AEF1',      '*B',
    '*C1e',   '*AE6e',   '*EF1e',      '*A', '*ACEG1e',    '*A6e',   '*AE1e',
   '*AC1e',     '*A5',    '*BE5']
Length: 24, dtype: str


## Flow rate method for electricity intensity

In [106]:

path = "C:/Users/skar/repos/wwtp_energy_comparison/input_data/flow_method/"
df1 = df[['CWNS_ID', 'TT_IDENTIFIED', 'PREDICTED_WRRF_TT_CODE', 'ACTUAL_FLOW_AVG', 
          'Electricity_consumed_onsite_kWh_per_m3',
          'Electricity_purchased_from_utility_kWh_per_m3',
          'Electricity_produced_onsite_kWh_per_m3',
          ]].copy()

# Assign flow categories based on EPRI flow categories
"""
LESS THAN 2
2 TO 4
4 TO 7
7 TO 16
16 TO 46
46 TO 100
100 AND ABOVE
"""
def assign_flow_category(flow_rate):
    if flow_rate < 2:
        return 'LESS THAN 2'
    elif flow_rate >= 2 and flow_rate < 4:
        return '2 TO 4'
    elif flow_rate >= 4 and flow_rate < 7:
        return '4 TO 7'
    elif flow_rate >= 7 and flow_rate < 16:
        return '7 TO 16'
    elif flow_rate >= 16 and flow_rate < 46:
        return '16 TO 46'
    elif flow_rate >= 46 and flow_rate < 100:
        return '46 TO 100'
    else:
        return '100 AND ABOVE'
df1['FLOW_CAT_MGD'] = df1['ACTUAL_FLOW_AVG'].apply(assign_flow_category)

#import electricity intensity values based on flow rate from EPRI
el_flow = pd.read_excel(path + 'epri_flow_ei.xlsx', sheet_name = 'EPRI_FLOW_EI')

# Merge electricity intensity values with flow categories
df1 = df1.merge(el_flow, how = 'left', left_on = 'FLOW_CAT_MGD', right_on = 'Average Daily Flow (MGD)')

# Save output
df1.to_csv(os.path.join(path_out, 'Electricity_by_flow_method.csv'), index = False)

## Effluent Treatment level methods A and B for electricity intensity

In [72]:
# TTs with E, F, G are assigned 'Advanced Treatment'
# TTs with just A or B is assigned 'Primmary' and 'Advanced Primary' respectively
# TTs with sludge treatment or incineration are assigned 'Secondary'

dict_treatment_levels = {
    '*AG2' : 'Advanced Treatment', 
    '*B6, *B' : 'Secondary',  
    '*AEF2e' : 'Advanced Treatment',    
    '*AE5' : 'Advanced Treatment',    
    '*AF2' : 'Advanced Treatment',    
    '*CE5' : 'Advanced Treatment',    
    '*B1e' : 'Secondary',
    '*EF1' : 'Advanced Treatment',    
    '*A1e' : 'Secondary',    
    '*AF1' : 'Advanced Treatment',     
    '*A1' : 'Secondary',     
    '*AEF1' : 'Advanced Treatment',
    '*B' : 'Advanced Primary',    
    '*C1e' : 'Secondary',   
    '*AE6e' : 'Advanced Treatment',   
    '*EF1e' : 'Advanced Treatment',      
    '*A' : 'Primary', 
    '*ACEG1e' : 'Advanced Treatment',    
    '*A6e' : 'Secondary',   
    '*AE1e' : 'Advanced Treatment',
    '*AC1e' : 'Advanced Treatment',     
    '*A5' : 'Secondary',    
    '*BE5' : 'Advanced Treatment'
}

df2 = df[['CWNS_ID', 'TT_IDENTIFIED', 'PREDICTED_WRRF_TT_CODE', 'ACTUAL_FLOW_AVG', 
          'BOD5_AVG',
          'Electricity_consumed_onsite_kWh_per_m3',
          'Electricity_purchased_from_utility_kWh_per_m3',
          'Electricity_produced_onsite_kWh_per_m3',
          ]].copy()
# remove rows with missing PREDICTED_WRRF_TT_CODE
df2 = df2[df2['PREDICTED_WRRF_TT_CODE'].notna()]

def assign_treatment_level(tt_code):
    for key in dict_treatment_levels.keys():
        if key in tt_code:
            return dict_treatment_levels[key]
    return 'Unknown'
df2['Treatment_Level'] = df2['PREDICTED_WRRF_TT_CODE'].apply(assign_treatment_level)

# Check on Primary assignment with BOD concentraiton for each row of df2
mask_primary = df2['Treatment_Level'] == 'Primary'
mask_low_bod = df2['BOD5_AVG'] < 45

df2.loc[mask_primary & mask_low_bod, 'Treatment_Level'] = 'Primary (45mg/l< BOD)'
df2.loc[mask_primary & ~mask_low_bod, 'Treatment_Level'] = 'Advanced Primary'

# Read Electricity intensity data
path = 'C:\\Users\\skar\\repos\\wwtp_energy_comparison\\input_data\\effluent_methods\\'
el_treatment_A = pd.read_excel(path + 'epri_effluent_a_ei.xlsx', sheet_name = 'Sheet1')
el_treatment_B = pd.read_excel(path + 'li_effluent_b_ei.xlsx', sheet_name = 'Sheet1')

# Merge electricity intensity values with treatment levels
df2 = df2.merge(el_treatment_A, how = 'left', left_on = 'Treatment_Level', right_on = 'Effluent Treatment Level')
df2 = df2.drop(columns = ['Effluent Treatment Level'])
df2 = df2.rename(columns = {'Electricity Intensity (kWh/m3)':'Electricity Intensity by A method (kWh/m3)'})
                            
df2 = df2.merge(el_treatment_B, how = 'left', left_on = 'Treatment_Level', right_on = 'Effluent Treatment Level')
df2 = df2.drop(columns = ['Effluent Treatment Level'])
df2 = df2.rename(columns = {'Electricity Intensity (kWh/m3)':'Electricity Intensity by B method (kWh/m3)'})

# Save output
df2.to_csv(os.path.join(path_out, 'Electricity_by_effluent_methods_A_B.csv'), index = False)


## Treatment Train Configuration (T) Methods for electricity and natural gas intensities

In [73]:
# load electricity intensity values for treatment train method
path = 'C:\\Users\\skar\\repos\\wwtp_energy_comparison\\input_data\\configuration_methods\\'
ei_config_T = pd.read_excel(path + 'configuration_t_ei.xlsx', sheet_name = 'All Trains (For Code)')
ei_config_T.rename(columns = {'Unnamed: 0':'Energy Intensity Type'}, inplace = True)

# Keep select relevant energy intensities
ei_config_T = ei_config_T[ei_config_T['Energy Intensity Type'].isin([
    'Total Electricity Usage [kWh/d] (including chemical production)', 
    'Total Natural Gas Usage [MJ/d] (including chemical production)'])]

# Transpose data frame
ei_config_T = ei_config_T.set_index('Energy Intensity Type').transpose().rename_axis('index').reset_index()
ei_config_T.rename(columns = {'index':'TT code'}, inplace = True)

# Update naming convention of TTs as represented in Hodson et al. analysis code.
TT_crosswalk = {
    '*AG2' : 'G2', 
    '*B6, *B' : 'O6',  
    '*AEF2e' : '',    # No code exists for F2e
    '*AE5' : '', # No code exists for E5    
    '*AF2' : '', # No code exists for F2    
    '*CE5' : '', # No code exists for E5    
    '*B1e' : 'O1E', 
    '*EF1' : 'I1',    
    '*A1e' : 'B1E',    
    '*AF1' : 'I1',     
    '*A1' : 'B1',     
    '*AEF1' : 'I1',
    '*B' : '', # No code exists for *B    
    '*C1e' : 'D1E',   
    '*AE6e' : '', # No code exists for E6e   
    '*EF1e' : 'I1E',      
    '*A' : '', # No code exists for *A 
    '*ACEG1e' : 'G1E',    
    '*A6e' : '', # No code exists for *A6e   
    '*AE1e' : 'F1E',
    '*AC1e' : 'D1E',     
    '*A5' : 'B5',    
    '*BE5' : '' # No code exists for E5
}

df3 = df[['CWNS_ID', 'TT_IDENTIFIED', 'PREDICTED_WRRF_TT_CODE', 'ACTUAL_FLOW_AVG',
          'Electricity_consumed_onsite_kWh_per_m3',
          'Electricity_purchased_from_utility_kWh_per_m3',
          'Electricity_produced_onsite_kWh_per_m3',
          'NATURAL_GAS_PURCHASED_onsite_MJ_per_m3'
          ]].copy()
# remove rows with missing PREDICTED_WRRF_TT_CODE
df3 = df3[df3['PREDICTED_WRRF_TT_CODE'].notna()]

def assign_tt_code(tt_code):
    for key in TT_crosswalk.keys():
        if key in tt_code:
            return TT_crosswalk[key]
    return 'TT crosswalk code not available'
df3['Treatment_Train_Code'] = df3['PREDICTED_WRRF_TT_CODE'].apply(assign_tt_code)

# Merge electricity intensity values with treatment train codes
df3 = df3.merge(ei_config_T, how = 'left', left_on = 'Treatment_Train_Code', right_on = 'TT code')
df3 = df3.drop(columns = ['TT code'])

# Convert kWh/d electricity to kWh_per_m3 using ACTUAL_FLOW_AVG which is in MGD. 1 MGD = 3785.41 m3/d
df3['Total Electricity Usage [kWh/m3] (including chemical production)'] = df3['Total Electricity Usage [kWh/d] (including chemical production)'] / (df3['ACTUAL_FLOW_AVG'] * 3785.41)

# Convert MJ/d natural gas to MJ/m3 using ACTUAL_FLOW_AVG which is in MGD. 1 MGD = 3785.41 m3/d
df3['Total Natural Gas Usage [MJ/m3] (including chemical production)'] = df3['Total Natural Gas Usage [MJ/d] (including chemical production)'] / (df3['ACTUAL_FLOW_AVG'] * 3785.41)

# Drop columns in kWh/d and MJ/d
df3 = df3.drop(columns = ['Total Electricity Usage [kWh/d] (including chemical production)', 
                          'Total Natural Gas Usage [MJ/d] (including chemical production)'])

# Save output
df3.to_csv(os.path.join(path_out, 'Energy_by_TT_config_T_method.csv'), index = False)


## Treatment Train Configuration (BP) Method for electricity and natural gas intensities

In [74]:
# load electricity intensity values for treatment train method
path = 'C:\\Users\\skar\\repos\\wwtp_energy_comparison\\input_data\\configuration_methods\\'
ei_config_T = pd.read_excel(path + 'configuration_bp_ei.xlsx', sheet_name = 'All Trains (For Code)')
ei_config_T.rename(columns = {'Unnamed: 0':'Energy Intensity Type'}, inplace = True)

# Keep select relevant energy intensities
ei_config_T = ei_config_T[ei_config_T['Energy Intensity Type'].isin([
    'Total Electricity Usage [kWh/d] (including chemical production)', 
    'Total Natural Gas Usage [MJ/d] (including chemical production)'])]

# Transpose data frame
ei_config_T = ei_config_T.set_index('Energy Intensity Type').transpose().rename_axis('index').reset_index()
ei_config_T.rename(columns = {'index':'TT code'}, inplace = True)

# Update naming convention of TTs as represented in Hodson et al. analysis code.
TT_crosswalk = {
    '*AG2' : 'G2', 
    '*B6, *B' : 'O6',  
    '*AEF2e' : '',    # No code exists for F2e
    '*AE5' : '', # No code exists for E5    
    '*AF2' : '', # No code exists for F2    
    '*CE5' : '', # No code exists for E5    
    '*B1e' : 'O1E', 
    '*EF1' : 'I1',    
    '*A1e' : 'B1E',    
    '*AF1' : 'I1',     
    '*A1' : 'B1',     
    '*AEF1' : 'I1',
    '*B' : '', # No code exists for *B    
    '*C1e' : 'D1E',   
    '*AE6e' : '', # No code exists for E6e   
    '*EF1e' : 'I1E',      
    '*A' : '', # No code exists for *A 
    '*ACEG1e' : 'G1E',    
    '*A6e' : '', # No code exists for *A6e   
    '*AE1e' : 'F1E',
    '*AC1e' : 'D1E',     
    '*A5' : 'B5',    
    '*BE5' : '' # No code exists for E5
}

df3 = df[['CWNS_ID', 'TT_IDENTIFIED', 'PREDICTED_WRRF_TT_CODE', 'ACTUAL_FLOW_AVG',
          'Electricity_consumed_onsite_kWh_per_m3',
          'Electricity_purchased_from_utility_kWh_per_m3',
          'Electricity_produced_onsite_kWh_per_m3',
          'NATURAL_GAS_PURCHASED_onsite_MJ_per_m3'
          ]].copy()
# remove rows with missing PREDICTED_WRRF_TT_CODE
df3 = df3[df3['PREDICTED_WRRF_TT_CODE'].notna()]

def assign_tt_code(tt_code):
    for key in TT_crosswalk.keys():
        if key in tt_code:
            return TT_crosswalk[key]
    return 'TT crosswalk code not available'
df3['Treatment_Train_Code'] = df3['PREDICTED_WRRF_TT_CODE'].apply(assign_tt_code)

# Merge electricity intensity values with treatment train codes
df3 = df3.merge(ei_config_T, how = 'left', left_on = 'Treatment_Train_Code', right_on = 'TT code')
df3 = df3.drop(columns = ['TT code'])

# Convert kWh/d electricity to kWh_per_m3 using ACTUAL_FLOW_AVG which is in MGD. 1 MGD = 3785.41 m3/d
df3['Total Electricity Usage [kWh/m3] (including chemical production)'] = df3['Total Electricity Usage [kWh/d] (including chemical production)'] / (df3['ACTUAL_FLOW_AVG'] * 3785.41)

# Convert MJ/d natural gas to MJ/m3 using ACTUAL_FLOW_AVG which is in MGD. 1 MGD = 3785.41 m3/d
df3['Total Natural Gas Usage [MJ/m3] (including chemical production)'] = df3['Total Natural Gas Usage [MJ/d] (including chemical production)'] / (df3['ACTUAL_FLOW_AVG'] * 3785.41)

# Drop columns in kWh/d and MJ/d
df3 = df3.drop(columns = ['Total Electricity Usage [kWh/d] (including chemical production)', 
                          'Total Natural Gas Usage [MJ/d] (including chemical production)'])

# Save output
df3.to_csv(os.path.join(path_out, 'Energy_by_TT_config_BP_method.csv'), index = False)


## Unit Process A Method for electricity intensity

In [105]:
#import intensity values for key unit processes from (Plappally and Leinhard, 2012) and drop duplicates
path = 'C:\\Users\\skar\\repos\\wwtp_energy_comparison\\input_data\\process_methods\\'
ei_UP_A = pd.read_excel(path + 'process_a_ei.xlsx')
ei_UP_A.drop_duplicates(subset = 'Unit Process Name', inplace = True)

UP_crosswalk = {
'*' : 'Grit Removal',
'A' : 'Activated Sludge, Conventional',
'B' : 'Activated Sludge, Pure Oxygen',
'C' : 'Trickling Filter, Rock Media',
'E' : 'Biological Nitrification - Separate Stage',
'F' : 'Activated Sludge, With Biological Denitrification', # When A and F both are present, need to only consider F energy
'G' : 'Phosphorus Removal, Biological',
'e' : '',
'1' : 'Biosolids Anaerobic Digestion, Thermophilic',
'2' : 'Biosolids Anaerobic Digestion, Thermophilic',
'5' : '', # Need to recheck literature and estimate energy for this process
'6' : '', # Need to recheck literature and estimate energy for this process
}

df4 = df[['CWNS_ID', 'TT_IDENTIFIED', 'PREDICTED_WRRF_TT_CODE', 'ACTUAL_FLOW_AVG',
          'Electricity_consumed_onsite_kWh_per_m3',
          'Electricity_purchased_from_utility_kWh_per_m3',
          'Electricity_produced_onsite_kWh_per_m3',
          'NATURAL_GAS_PURCHASED_onsite_MJ_per_m3'
          ]].copy()
# remove rows with missing PREDICTED_WRRF_TT_CODE
df4 = df4[df4['PREDICTED_WRRF_TT_CODE'].notna()]

# replace PREDICTED_WRRF_TT_CODE '*B6, *B' with '*B6'
df4['PREDICTED_WRRF_TT_CODE'] = df4['PREDICTED_WRRF_TT_CODE'].replace('*B6, *B', '*B6')

# Create column for each unit process in keys of UP_crosswalk and assign 1 if the process character code is present in PREDICTED_WRRF_TT_CODE and 0 if not
for key in UP_crosswalk.keys():
    # if UP_crosswalk[key] is empty, don't create column for that unit process
    if UP_crosswalk[key]:
        df4[UP_crosswalk[key]] = df4['PREDICTED_WRRF_TT_CODE'].apply(lambda x: 1 if key in x else 0)

UP_names = list(set(UP_crosswalk.values()))

# From column 8 onwards in df4, multiply the energy intensity from ei_UP_A_sub with the column value (0 or 1)
for up in UP_names:
    if up:
        energy_intensity = ei_UP_A[ei_UP_A['Unit Process Name'] == up]['Electricity Intensity (Average) (kWh/m3)'].iloc[0]
        print (up, energy_intensity, 'kWh/m3')
        df4[up + ' Electricity Intensity (kWh/m3)'] = df4[up] * energy_intensity
    
# When A and F both are present, need to only consider F energy
mask_AF = (df4['Activated Sludge, Conventional'] == 1) & (df4['Activated Sludge, With Biological Denitrification'] == 1)
df4.loc[mask_AF, 'Activated Sludge, Conventional Electricity Intensity (kWh/m3)'] = 0

# Sum up energy intensity for all unit processes to get total energy intensity for each row
df4['Total Electricity Intensity (kWh/m3)'] = df4[[up + ' Electricity Intensity (kWh/m3)' for up in UP_names if up]].sum(axis = 1)

# Save output
df4.to_csv(os.path.join(path_out, 'Energy_by_unit_process_A_method.csv'), index = False)

Activated Sludge, Conventional 0.465 kWh/m3
Activated Sludge, Pure Oxygen 0.465 kWh/m3
Activated Sludge, With Biological Denitrification 0.55 kWh/m3
Biosolids Anaerobic Digestion, Thermophilic 0.265 kWh/m3
Phosphorus Removal, Biological 0.1 kWh/m3
Trickling Filter, Rock Media 0.321 kWh/m3
Biological Nitrification - Separate Stage 0.085 kWh/m3
Grit Removal 0.015 kWh/m3


## Unit Process B Method for electricity intensity

In [ ]:
UP_crosswalk = {
    '*AG2' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion', 'Thermal drying'],
    '*B6, *B' : ['Grit removal, aerated', ],
    '*AEF2e' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion', 'Thermal drying', 'Energy recovery (from biogas combustion)'],
    '*AE5' : ['Grit removal, aerated', 'Aeration with BNR', ],    
    '*AF2' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion', 'Thermal drying'],   
    '*CE5' : ['Grit removal, aerated', 'Trickling filters', 'Aeration with BNR', ],     
    '*B1e' : ['Grit removal, aerated', 'Anaerobic digestion', 'Energy recovery (from biogas combustion)'], 
    '*EF1' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion'],    
    '*A1e' : ['Grit removal, aerated', 'Anaerobic digestion', 'Energy recovery (from biogas combustion)'],    
    '*AF1' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion'], 
    '*A1' : ['Grit removal, aerated', 'Anaerobic digestion'],     
    '*AEF1' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion'],
    '*B' : ['Grit removal, aerated', ],    
    '*C1e' : ['Grit removal, aerated', 'Trickling filters', 'Anaerobic digestion', 'Energy recovery (from biogas combustion)'],   
    '*AE6e' : ['Grit removal, aerated', 'Aeration with BNR', 'Energy recovery (from biogas combustion)'],   
    '*EF1e' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion', 'Energy recovery (from biogas combustion)'],      
    '*A' : ['Grit removal, aerated', ],
    '*ACEG1e' : ['Grit removal, aerated', 'Trickling filters', 'Aeration with BNR', 'Anaerobic digestion', 'Energy recovery (from biogas combustion)'],    
    '*A6e' : ['Grit removal, aerated', 'Energy recovery (from biogas combustion)'],   
    '*AE1e' : ['Grit removal, aerated', 'Aeration with BNR', 'Anaerobic digestion', 'Energy recovery (from biogas combustion)'],
    '*AC1e' : ['Grit removal, aerated', 'Trickling filters', 'Anaerobic digestion', 'Energy recovery (from biogas combustion)'],     
    '*A5' : ['Grit removal, aerated', ],    
    '*BE5' : ['Grit removal, aerated', 'Aeration with BNR', ],
}
# above processes does not include energy requirement for 5 and 6, incineration based approaches. Need to check literature for such values.

# add 'Total baseload (wastewater pumping, odor, utility water, non-process loads)' and 'Nonprocess loads (buildings, lighting, computers, pneumatics, etc.)' to all key-values
for key in UP_crosswalk.keys():
    UP_crosswalk[key].append('Total baseload (wastewater pumping, odor, utility water, non-process loads)')
    UP_crosswalk[key].append('Nonprocess loads (buildings, lighting, computers, pneumatics, etc.)')

# next steps for next week
# map UP to facilities
# import mechanical solids treatment from survey table and include the UP for those
# identify nearest flow rate for faciliteis
# Calculate intensity by UP